# Preprocess code 2 - A2 retailer
## Universidad ICESI 
### David Mauricio Orozco Rios
### author: Davoroz06 - IG

In [ ]:
# libraries
import pandas as pd
import numpy as np

# project paths and shared helpers (see src/config.py)
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import *  # wd, wd_dp, wd_db, wd_dpr, wd_re, wd_rp, wd_rt ...
from utils import homogenize_text

Load data:

In [ ]:
data = pd.read_csv(wd_dp + "A2_raw.csv")
len(data)

Join columns with same meaning

In [ ]:
data["descripcion"] = data['descripcion'].fillna(data['Descripcion'])
data["precio"] = data['precio'].fillna(data['low'])
data["precio"] = data['precio'].fillna(data['high'])
data["precio"] = data['precio'].fillna(data['oferta'])
data["fecha"] = data["Fecha"].fillna(data["dia"])
data["link"] = data["link"].fillna(data["Link"])
data["link"] = data["link"].fillna(data["Link"])
data["palabra"] = data["palabra"].fillna(data["item"])

Filter empty products

In [ ]:
data = data[~data['descripcion'].isna()]
len(data)

Fill searched word 

In [ ]:
data['palabra'] = data['palabra'].fillna(data['Link'].str.extract(r"https?://[^/]+/(.*)\?").squeeze())

String homogenize

In [ ]:
data["palabra"] = data["palabra"].apply(homogenize_text)
data["descripcion"] = data["descripcion"].apply(homogenize_text)

Price homogenize

In [ ]:
data["precio"] = data["precio"].astype("str")
data["precio"] = data["precio"].str.replace("\.0$", "", regex=True)
data["precio"] = data["precio"].str.replace("[a-zA-Z]", "", regex=True)
data["precio"] = data["precio"].str.replace(".", "")
data["precio"] = data["precio"].str.replace(",", ".")
data["precio"] = data["precio"].str.replace("$", "")

Filter empty price products

In [ ]:
data = data[data["precio"].str.len()!=0]
data = data[~data['precio'].isna()]

Price to numeric

In [ ]:
data["precio"] = data["precio"].astype("float")

In [ ]:
data.columns

Select columns

In [ ]:
data = data[["fecha", "descripcion", "precio", "palabra"]]
data["tienda"] = "A2"

Summarize duplicate prices

In [ ]:
data = data.groupby(['fecha', 'descripcion', 'tienda']).agg({'precio': 'min'}).reset_index()

Show data

In [ ]:
display(data)

Save preprocess data

In [ ]:
pd.DataFrame(data.to_csv(wd_db+"A2_clean.csv"))